# MIT 8339 – BI & Decision Support Systems (Virtual Lab)
**Customer Retention and Performance Intelligence for a Nigerian Commercial Bank**

Student: Blossom Idigbo | Matric No: 2025/B/MIT/0906 | Email: b.idigbo1616@miva.edu.ng

Dataset: Goyal, S. (2020). *Credit Card Customers (BankChurners)* [Data set]. Kaggle.
Place `BankChurners.csv` in the same folder as this notebook and run all cells (Python 3, pandas, numpy, matplotlib).

## Task 1 – Data collection and KPI definitions
Decision to support: *which customer groups are at highest risk of closing their card account, and where should retention spend be targeted?*

In [1]:
import pandas as pd, numpy as np, json
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 40)
raw = pd.read_csv("BankChurners.csv")
print("Raw shape:", raw.shape)
raw.head(3)

Raw shape: (10127, 23)


,CLIENTNUM,Attrition_Flag,Customer_Age,Gender,Dependent_count,Education_Level,Marital_Status,Income_Category,Card_Category,Months_on_book,Total_Relationship_Count,Months_Inactive_12_mon,Contacts_Count_12_mon,Credit_Limit,Total_Revolving_Bal,Avg_Open_To_Buy,Total_Amt_Chng_Q4_Q1,Total_Trans_Amt,Total_Trans_Ct,Total_Ct_Chng_Q4_Q1,Avg_Utilization_Ratio,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_1,Naive_Bayes_Classifier_Attrition_Flag_Card_Category_Contacts_Count_12_mon_Dependent_count_Education_Level_Months_Inactive_12_mon_2
0,768805383,Existing Customer,45,M,3,High School,Married,$60K - $80K,Blue,39,5,1,3,12691.0,777,11914.0,1.335,1144,42,1.625,0.061,0.000093,0.99991
1,818770008,Existing Customer,49,F,5,Graduate,Single,Less than $40K,Blue,44,6,1,2,8256.0,864,7392.0,1.541,1291,33,3.714,0.105,0.000057,0.99994
2,713982108,Existing Customer,51,M,3,Graduate,Married,$80K - $120K,Blue,36,4,1,0,3418.0,0,3418.0,2.594,1887,20,2.333,0.000,0.000021,0.99998


## Task 2 – Data profiling (BEFORE cleaning)

In [2]:
def quality_report(df, label):
    rep = {
        "stage": label,
        "rows": len(df),
        "columns": df.shape[1],
        "missing_cells": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_CLIENTNUM": int(df["CLIENTNUM"].duplicated().sum()),
        "'Unknown' labels (Education/Marital/Income)": int(sum((df[c] == "Unknown").sum() for c in ["Education_Level","Marital_Status","Income_Category"])),
        "constant/leaky model-output columns": int(sum(c.startswith("Naive_Bayes") for c in df.columns)),
    }
    return rep
before = quality_report(raw, "Before")
print(before)
print(raw.dtypes.value_counts())
for c in ["Attrition_Flag","Gender","Education_Level","Marital_Status","Income_Category","Card_Category"]:
    print(c, raw[c].value_counts().to_dict())

{'stage': 'Before', 'rows': 10127, 'columns': 23, 'missing_cells': 0, 'duplicate_rows': 0, 'duplicate_CLIENTNUM': 0, "'Unknown' labels (Education/Marital/Income)": 3380, 'constant/leaky model-output columns': 2}
int64      10
float64     7
str         6
Name: count, dtype: int64
Attrition_Flag {'Existing Customer': 8500, 'Attrited Customer': 1627}
Gender {'F': 5358, 'M': 4769}
Education_Level {'Graduate': 3128, 'High School': 2013, 'Unknown': 1519, 'Uneducated': 1487, 'College': 1013, 'Post-Graduate': 516, 'Doctorate': 451}
Marital_Status {'Married': 4687, 'Single': 3943, 'Unknown': 749, 'Divorced': 748}
Income_Category {'Less than $40K': 3561, '$40K - $60K': 1790, '$80K - $120K': 1535, '$60K - $80K': 1402, 'Unknown': 1112, '$120K +': 727}
Card_Category {'Blue': 9436, 'Silver': 555, 'Gold': 116, 'Platinum': 20}


In [3]:
# Outlier check (IQR rule) on numeric fields
num = ["Customer_Age","Credit_Limit","Total_Trans_Amt","Total_Trans_Ct","Total_Amt_Chng_Q4_Q1","Total_Ct_Chng_Q4_Q1","Total_Revolving_Bal","Avg_Utilization_Ratio"]
rows = []
for c in num:
    q1, q3 = raw[c].quantile([.25,.75]); iqr = q3-q1
    n = int(((raw[c] < q1-1.5*iqr) | (raw[c] > q3+1.5*iqr)).sum())
    rows.append([c, raw[c].min(), raw[c].max(), n, round(100*n/len(raw),1)])
outliers = pd.DataFrame(rows, columns=["field","min","max","IQR_outliers","pct"])
print(outliers.to_string(index=False))
# Consistency checks on business rules
print("Open-to-buy = limit - revolving balance holds:", bool(np.allclose(raw.Avg_Open_To_Buy, raw.Credit_Limit - raw.Total_Revolving_Bal)))
print("Utilisation = revolving bal / limit holds (±0.001):", bool(((raw.Total_Revolving_Bal/raw.Credit_Limit) - raw.Avg_Utilization_Ratio).abs().max() < 0.001))
print("Negative values anywhere:", bool((raw.select_dtypes("number") < 0).any().any()))

                field    min       max  IQR_outliers  pct
         Customer_Age   26.0    73.000             2  0.0
         Credit_Limit 1438.3 34516.000           984  9.7
      Total_Trans_Amt  510.0 18484.000           896  8.8
       Total_Trans_Ct   10.0   139.000             2  0.0
 Total_Amt_Chng_Q4_Q1    0.0     3.397           396  3.9
  Total_Ct_Chng_Q4_Q1    0.0     3.714           394  3.9
  Total_Revolving_Bal    0.0  2517.000             0  0.0
Avg_Utilization_Ratio    0.0     0.999             0  0.0
Open-to-buy = limit - revolving balance holds: True
Utilisation = revolving bal / limit holds (±0.001): True
Negative values anywhere: False


**Cleaning decisions (justified)**
1. **No missing cells, no duplicates** – verified; nothing to drop or impute.
2. **Two `Naive_Bayes_Classifier…` columns dropped** – they are outputs of a previous model that already encode the answer (target leakage) and are not business fields.
3. **"Unknown" in Education, Marital Status and Income** – kept as an explicit category rather than deleting ~30% of rows or guessing values; attrition for "Unknown" is similar to the overall rate, so it does not distort the analysis. A `*_Known` flag lets analysts exclude them.
4. **Outliers retained** – high credit limits, large transaction totals and Q4/Q1 change ratios are valid business values (premium customers / spending spikes, and the arithmetic checks above pass). Deleting them would remove exactly the high-value customers the bank cares about. They are flagged, not removed.
5. Labels standardised (snake_case column names, `Attrition_Flag` → binary `Attrited`), income labels given a sortable order.

In [4]:
df = raw.drop(columns=[c for c in raw.columns if c.startswith("Naive_Bayes")]).copy()
df.columns = [c.strip() for c in df.columns]
df["Attrited"] = (df["Attrition_Flag"] == "Attrited Customer").astype(int)
df["Gender"] = df["Gender"].map({"F":"Female","M":"Male"})

# derived analytical fields
df["Age_Band"] = pd.cut(df.Customer_Age, [0,29,39,49,59,120], labels=["<30","30-39","40-49","50-59","60+"])
inc_order = ["Less than $40K","$40K - $60K","$60K - $80K","$80K - $120K","$120K +","Unknown"]
df["Income_Category"] = pd.Categorical(df.Income_Category, inc_order, ordered=True)
df["Utilisation_Category"] = pd.cut(df.Avg_Utilization_Ratio, [-0.001,0.05,0.30,0.60,1.0], labels=["Very low (<5%)","Low (5-30%)","Medium (30-60%)","High (>60%)"])
df["Txn_Frequency_Category"] = pd.cut(df.Total_Trans_Ct, [0,40,60,80,200], labels=["Very low (≤40)","Low (41-60)","Medium (61-80)","High (>80)"])
df["Tenure_Band"] = pd.cut(df.Months_on_book, [0,24,36,48,100], labels=["≤24m","25-36m","37-48m","48m+"])
df["Customer_Value_Class"] = pd.qcut(df.Total_Trans_Amt, [0,.5,.8,1], labels=["Standard","Valuable","Premium"])
for c in ["Education_Level","Marital_Status","Income_Category"]:
    df[c+"_Known"] = (df[c].astype(str) != "Unknown").astype(int)
# outlier flags (kept, not removed)
for c in ["Credit_Limit","Total_Trans_Amt"]:
    q1,q3 = df[c].quantile([.25,.75]); i=q3-q1
    df[c+"_Outlier"] = ((df[c]<q1-1.5*i)|(df[c]>q3+1.5*i)).astype(int)

## Task 3 – Rule-based retention-risk score
Points are assigned from behaviours that the EDA below shows are strongly associated with attrition
(low transaction count, falling activity, very low utilisation, few products, many service contacts, inactivity).

In [5]:
s  = np.select([df.Total_Trans_Ct < 45, df.Total_Trans_Ct < 65], [2,1], 0)          # low transaction frequency
s += np.select([df.Total_Ct_Chng_Q4_Q1 < 0.5, df.Total_Ct_Chng_Q4_Q1 < 0.7], [2,1], 0)  # activity falling Q1→Q4
s += (df.Avg_Utilization_Ratio < 0.05).astype(int)       # card barely used
s += (df.Total_Relationship_Count <= 2).astype(int)      # few products held
s += (df.Contacts_Count_12_mon >= 3).astype(int)         # repeated contact with the bank
s += (df.Months_Inactive_12_mon >= 3).astype(int)        # inactive 3+ months
df["Risk_Score"] = s
df["Risk_Band"] = pd.cut(s, [-1,2,4,20], labels=["Low","Medium","High"])
print(df.Risk_Score.value_counts().sort_index().to_dict())
risk = df.groupby("Risk_Band", observed=True).agg(Customers=("Attrited","size"), Attrited=("Attrited","sum"), Attrition_Rate=("Attrited","mean"))
risk["Share_of_customers_%"] = (100*risk.Customers/len(df)).round(1)
risk["Share_of_all_attrition_%"] = (100*risk.Attrited/df.Attrited.sum()).round(1)
risk["Attrition_Rate"] = (100*risk.Attrition_Rate).round(1)
risk

{0: 523, 1: 1766, 2: 2649, 3: 2245, 4: 1377, 5: 811, 6: 509, 7: 200, 8: 47}


,Customers,Attrited,Attrition_Rate,Share_of_customers_%,Share_of_all_attrition_%
Risk_Band,,,,,
Low,4938,109,2.2,48.8,6.7
Medium,3622,516,14.2,35.8,31.7
High,1567,1002,63.9,15.5,61.6


## Data profiling (AFTER cleaning) and export

In [6]:
after = quality_report(df, "After")
after["columns"] = df.shape[1]
qc = pd.DataFrame([before, after]).set_index("stage").T
print(qc)
df.to_csv("BankChurners_cleaned.csv", index=False)
print("Saved BankChurners_cleaned.csv", df.shape)

stage                                        Before  After
rows                                          10127  10127
columns                                          23     34
missing_cells                                     0      0
duplicate_rows                                    0      0
duplicate_CLIENTNUM                               0      0
'Unknown' labels (Education/Marital/Income)    3380   3380
constant/leaky model-output columns               2      0
Saved BankChurners_cleaned.csv (10127, 34)


## Task 3 – KPIs (full dataset and by segment)

In [7]:
kpi = {
 "Total customers": len(df),
 "Attrition rate (%)": round(100*df.Attrited.mean(),2),
 "Attrited customers": int(df.Attrited.sum()),
 "Active customer ratio (%) (≤2 inactive months)": round(100*(df.Months_Inactive_12_mon<=2).mean(),1),
 "Average credit limit ($)": round(df.Credit_Limit.mean(),0),
 "Average utilisation ratio (%)": round(100*df.Avg_Utilization_Ratio.mean(),1),
 "Average total transaction amount ($)": round(df.Total_Trans_Amt.mean(),0),
 "Average transaction count": round(df.Total_Trans_Ct.mean(),1),
 "Average products held": round(df.Total_Relationship_Count.mean(),2),
 "Transaction value lost with attrited customers (% of total)": round(100*df.loc[df.Attrited==1,"Total_Trans_Amt"].sum()/df.Total_Trans_Amt.sum(),1),
}
kpis = pd.Series(kpi, name="Value"); print(kpis.to_string())

Total customers                                                10127.00
Attrition rate (%)                                                16.07
Attrited customers                                              1627.00
Active customer ratio (%) (≤2 inactive months)                    54.70
Average credit limit ($)                                        8632.00
Average utilisation ratio (%)                                     27.50
Average total transaction amount ($)                            4404.00
Average transaction count                                         64.90
Average products held                                              3.81
Transaction value lost with attrited customers (% of total)       11.30


In [8]:
pd.options.display.float_format = "{:,.1f}".format
summary = df[["Customer_Age","Credit_Limit","Total_Trans_Amt","Total_Trans_Ct","Avg_Utilization_Ratio","Months_Inactive_12_mon","Contacts_Count_12_mon","Total_Relationship_Count"]].describe().T
print(summary[["mean","std","min","50%","max"]])

def seg(col):
    g = df.groupby(col, observed=True).agg(Customers=("Attrited","size"), Attrition_Rate_pct=("Attrited","mean"),
        Avg_Credit_Limit=("Credit_Limit","mean"), Avg_Trans_Amt=("Total_Trans_Amt","mean"), Avg_Utilisation_pct=("Avg_Utilization_Ratio","mean"))
    g["Attrition_Rate_pct"]*=100; g["Avg_Utilisation_pct"]*=100
    return g
for c in ["Card_Category","Income_Category","Age_Band","Gender","Utilisation_Category","Txn_Frequency_Category","Total_Relationship_Count"]:
    print("\n--", c); print(seg(c).round(1).to_string())

                            mean     std     min     50%      max
Customer_Age                46.3     8.0    26.0    46.0     73.0
Credit_Limit             8,632.0 9,088.8 1,438.3 4,549.0 34,516.0
Total_Trans_Amt          4,404.1 3,397.1   510.0 3,899.0 18,484.0
Total_Trans_Ct              64.9    23.5    10.0    67.0    139.0
Avg_Utilization_Ratio        0.3     0.3     0.0     0.2      1.0
Months_Inactive_12_mon       2.3     1.0     0.0     2.0      6.0
Contacts_Count_12_mon        2.5     1.1     0.0     2.0      6.0
Total_Relationship_Count     3.8     1.6     1.0     4.0      6.0

-- Card_Category
               Customers  Attrition_Rate_pct  Avg_Credit_Limit  Avg_Trans_Amt  Avg_Utilisation_pct
Card_Category                                                                                     
Blue                9436                16.1           7,363.8        4,225.4                 29.1
Gold                 116                18.1          28,416.4        7,685.6              

        Customers  Attrition_Rate_pct  Avg_Credit_Limit  Avg_Trans_Amt  Avg_Utilisation_pct
Gender                                                                                     
Female       5358                17.4           5,023.9        4,324.3                 34.2
Male         4769                14.6          12,685.7        4,493.7                 20.0

-- Utilisation_Category
                      Customers  Attrition_Rate_pct  Avg_Credit_Limit  Avg_Trans_Amt  Avg_Utilisation_pct
Utilisation_Category                                                                                     
Very low (<5%)             3019                32.5          11,783.5        4,266.8                  0.7
Low (5-30%)                3234                 9.2          12,430.8        4,939.6                 15.2
Medium (30-60%)            2072                 8.1           3,422.2        4,342.2                 45.5
High (>60%)                1802                10.0           2,524.5        

                          Customers  Attrition_Rate_pct  Avg_Credit_Limit  Avg_Trans_Amt  Avg_Utilisation_pct
Total_Relationship_Count                                                                                     
1                               910                25.6          10,185.7        7,252.9                 24.6
2                              1243                27.8          10,294.1        7,019.1                 23.1
3                              2305                17.4           8,311.2        3,936.3                 27.1
4                              1912                11.8           8,212.2        3,755.0                 28.8
5                              1891                12.0           7,945.2        3,515.3                 29.3
6                              1866                10.5           8,289.3        3,416.5                 29.1


### Attrited vs Existing – eight variables

In [9]:
cmp_vars = ["Total_Trans_Ct","Total_Trans_Amt","Total_Ct_Chng_Q4_Q1","Avg_Utilization_Ratio","Total_Revolving_Bal","Total_Relationship_Count","Contacts_Count_12_mon","Months_Inactive_12_mon","Credit_Limit","Customer_Age"]
cmp = df.groupby("Attrition_Flag")[cmp_vars].mean().T
cmp["Difference_%"] = 100*(cmp["Attrited Customer"]/cmp["Existing Customer"]-1)
from math import sqrt
def cohen_d(a,b): return (a.mean()-b.mean())/sqrt((a.var()+b.var())/2)
cmp["Effect_size_d"] = [cohen_d(df.loc[df.Attrited==1,v], df.loc[df.Attrited==0,v]) for v in cmp_vars]
print(cmp.round(2).sort_values("Effect_size_d", key=abs, ascending=False).to_string())

Attrition_Flag            Attrited Customer  Existing Customer  Difference_%  Effect_size_d
Total_Trans_Ct                         44.9               68.7         -34.6           -1.2
Total_Ct_Chng_Q4_Q1                     0.6                0.7         -25.3           -0.8
Total_Revolving_Bal                   672.8            1,256.6         -46.5           -0.7
Contacts_Count_12_mon                   3.0                2.4          26.1            0.6
Total_Trans_Amt                     3,095.0            4,654.7         -33.5           -0.5
Avg_Utilization_Ratio                   0.2                0.3         -45.2           -0.5
Months_Inactive_12_mon                  2.7                2.3          18.4            0.4
Total_Relationship_Count                3.3                3.9         -16.2           -0.4
Credit_Limit                        8,136.0            8,726.9          -6.8           -0.1
Customer_Age                           46.7               46.3           0.9    

### Patterns explaining attrition

In [10]:
print("Attrition by months inactive:\n", df.groupby("Months_Inactive_12_mon").Attrited.agg(["mean","size"]).round(3).T.to_string())
print("\nAttrition by contacts:\n", df.groupby("Contacts_Count_12_mon").Attrited.agg(["mean","size"]).round(3).T.to_string())
print("\nAttrition by product count:\n", df.groupby("Total_Relationship_Count").Attrited.agg(["mean","size"]).round(3).T.to_string())
low_act = (df.Total_Trans_Ct<45)
print("\nTrans count <45 -> attrition %.1f%% (n=%d); >=45 -> %.1f%%" % (100*df[low_act].Attrited.mean(), low_act.sum(), 100*df[~low_act].Attrited.mean()))
print("Unknown-income attrition %.1f%% vs known %.1f%%" % (100*df[df.Income_Category=="Unknown"].Attrited.mean(), 100*df[df.Income_Category!="Unknown"].Attrited.mean()))

Attrition by months inactive:
 Months_Inactive_12_mon    0       1       2       3     4     5     6
mean                    0.5     0.0     0.2     0.2   0.3   0.2   0.2
size                   29.0 2,233.0 3,282.0 3,846.0 435.0 178.0 124.0

Attrition by contacts:
 Contacts_Count_12_mon     0       1       2       3       4     5    6
mean                    0.0     0.1     0.1     0.2     0.2   0.3  1.0
size                  399.0 1,499.0 3,227.0 3,380.0 1,392.0 176.0 54.0

Attrition by product count:
 Total_Relationship_Count     1       2       3       4       5       6
mean                       0.3     0.3     0.2     0.1     0.1     0.1
size                     910.0 1,243.0 2,305.0 1,912.0 1,891.0 1,866.0

Trans count <45 -> attrition 36.5% (n=2482); >=45 -> 9.4%
Unknown-income attrition 16.8% vs known 16.0%


### Validating the risk score (in-sample)

In [11]:
from itertools import product
y = df.Attrited
hi = df.Risk_Band=="High"
prec = y[hi].mean(); rec = y[hi].sum()/y.sum()
print(f"High band: {hi.sum()} customers, precision {prec:.1%}, recall {rec:.1%}, lift {prec/y.mean():.1f}x")
hm = df.Risk_Band.isin(["High","Medium"])
print(f"High+Medium: {hm.sum()} customers, precision {y[hm].mean():.1%}, recall {y[hm].sum()/y.sum():.1%}")
# simple AUC via rank statistic
from scipy.stats import rankdata
r = rankdata(df.Risk_Score); n1=y.sum(); n0=len(y)-n1
print("AUC of score: %.3f" % ((r[y==1].sum()-n1*(n1+1)/2)/(n1*n0)))
# value-at-risk
print("Avg annual transaction value of High-band customers: $%.0f" % df[hi].Total_Trans_Amt.mean())

High band: 1567 customers, precision 63.9%, recall 61.6%, lift 4.0x


High+Medium: 5189 customers, precision 29.3%, recall 93.3%


AUC of score: 0.882
Avg annual transaction value of High-band customers: $2166
